# Multi-Document RAG with Chat Memory
Humanli.ai Assignment 3. 

In [59]:
%%capture
!pip install -q -U langchain-core langchain-community langchain-huggingface langchain-text-splitters faiss-cpu rank_bm25 pypdf sentence-transformers transformers accelerate requests pdfplumber reportlab "pillow<13"

## 1. Config

In [60]:
import os, re, json, time, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # cleaner output

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path(".")
PDF_DIR = WORK / "data" / "pdfs"
STORE_DIR = WORK / "rag_store"
PDF_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    embed_model="BAAI/bge-small-en-v1.5",
    rerank_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    llm_model="Qwen/Qwen2.5-3B-Instruct",
    chunk_size=1000,
    chunk_overlap=150,
    k_dense=15,
    k_bm25=15,
    pool=20,                # fused candidates sent to the reranker
    k_final=5,              # chunks given to the llm
    rerank_min_score=-4.0,  # below this = not in the documents
    max_new_tokens=450,
    keep_turns=4,           # recent turns kept as is, older ones get summarised
    tables=True,            # also extract tables with pdfplumber (slower, needed for reports)
    verify=True,            # second guard: refuse when p_yes (section 9) is below verify_min
    verify_min=0.001,       # eval: unanswerable p_yes <= 1e-8, answerable >= 0.04
    # optional: phrases that tell us which document a question is about
    # (documents without aliases still work through the normal hybrid search)
    aliases={
        "attention_is_all_you_need.pdf": ["attention is all you need", "transformer paper"],
        "rag_lewis_2020.pdf":            ["original rag", "lewis"],
        "llama2_technical_report.pdf":   ["llama 2", "llama2"],
        "rag_survey_gao_2023.pdf":       ["rag survey", "the survey", "gao"],
        "lora_hu_2021.pdf":              ["lora paper"],
        "acme_annual_report.pdf":        ["acme"],
    },
)
NOT_FOUND_MSG = "I could not find this information in the uploaded documents."

## 2. Documents

In [61]:
import requests

# dataset: 5 public arXiv papers (add your own pdfs in data/pdfs or via Kaggle Add Data)
PAPERS = {
    "attention_is_all_you_need.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_lewis_2020.pdf":            "https://arxiv.org/pdf/2005.11401",
    "llama2_technical_report.pdf":   "https://arxiv.org/pdf/2307.09288",
    "rag_survey_gao_2023.pdf":       "https://arxiv.org/pdf/2312.10997",
    "lora_hu_2021.pdf":              "https://arxiv.org/pdf/2106.09685",
}
HEADERS = {"User-Agent": "Mozilla/5.0 (research-rag-assignment)"}

for name, url in PAPERS.items():
    dest = PDF_DIR / name
    if dest.exists() and dest.stat().st_size > 50_000:
        continue
    try:
        r = requests.get(url, headers=HEADERS, timeout=90)
        if r.content[:4] == b"%PDF":
            dest.write_bytes(r.content)
            print("downloaded", name, f"{len(r.content)/1e6:.1f} MB")
        else:
            print("not a pdf, skipped:", name)
    except Exception as e:
        print("failed", name, e)

# pdfs attached with Add Data
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").rglob("*.pdf"):
        shutil.copy(p, PDF_DIR / p.name)

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"\n{len(pdfs)} PDFs ready:")
for p in pdfs: print(" -", p.name)


6 PDFs ready:
 - acme_annual_report.pdf
 - attention_is_all_you_need.pdf
 - llama2_technical_report.pdf
 - lora_hu_2021.pdf
 - rag_lewis_2020.pdf
 - rag_survey_gao_2023.pdf


## 2b. Small financial report (table test)

In [62]:
# synthetic report with a table, to test numbers and tables (research papers have few)
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, PageBreak, Spacer

acme = PDF_DIR / "acme_annual_report.pdf"
st = getSampleStyleSheet()
table = Table([["Metric (INR crore)", "2021", "2022", "2023"],
               ["Revenue", "1,820", "2,145", "2,510"],
               ["Net profit", "212", "268", "331"],
               ["EBITDA margin", "14.2%", "15.1%", "16.0%"]])
table.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.grey)]))
intro = ("Acme Industries manufactures industrial sensors and is headquartered in Pune. "
         "The Chief Executive Officer is Meera Rao. The company had 4,200 employees at year end. ") * 2
SimpleDocTemplate(str(acme), pagesize=A4).build([
    Paragraph("Acme Industries Annual Report 2023", st["Title"]), Paragraph(intro, st["Normal"]), PageBreak(),
    Paragraph("Consolidated financial summary", st["Heading2"]), table, Spacer(1, 12),
    Paragraph("Revenue grew 17 percent in 2023 driven by exports.", st["Normal"])])
pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(len(pdfs), "PDFs now")

6 PDFs now


## 3. Embedding and rerank models

In [63]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

emb = HuggingFaceEmbeddings(
    model_name=CFG["embed_model"],
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True})
reranker = CrossEncoder(CFG["rerank_model"], device=device)

device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 4. Index and retrieval

In [64]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

STOP = set("a an the of and or to in on for with is are was were be by as at from that this it its what which who how does do did".split())

def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

def clean(t):
    t = re.sub(r"-\n(\w)", r"\1", t)       # join hyphenated line breaks
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def is_reference_chunk(t):
    # bibliography chunks hurt retrieval, drop them
    low = t.lower()
    numbered = len(re.findall(r"^\s*\[\d+\]\s", t, flags=re.M))
    years = len(re.findall(r"\b(19|20)\d{2}\b", t))
    return numbered >= 3 or low.count("arxiv preprint") >= 2 or (low.count("et al.") >= 6 and years >= 6)

# comparison questions need chunks from more than one document
MULTI_RE = re.compile(r"\b(compare|comparison|contrast|differences?|differs?|versus|vs\.?|both|across|"
                      r"each of|all (the )?(papers|documents))\b", re.I)


class DocumentIndex:
    def __init__(self, emb, reranker, cfg):
        self.emb, self.reranker, self.cfg = emb, reranker, cfg
        self.aliases = cfg.get("aliases", {})
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"],
            separators=["\n\n", "\n", ". ", " ", ""])
        self.chunks, self.sources = [], []
        self.vs = self.bm25 = None

    # ingestion
    def _load(self, path):
        pages = []
        for d in PyPDFLoader(str(path)).load():
            text = clean(d.page_content)
            if len(text) < 50:             # blank or figure-only page
                continue
            pages.append(Document(page_content=text,
                                  metadata={"source": path.name, "page": int(d.metadata.get("page", 0)) + 1}))
        if self.cfg.get("tables"):
            pages += self._tables(path)
        return pages

    def _tables(self, path):
        # pypdf scrambles tables (one cell per line), pdfplumber keeps the rows
        out = []
        try:
            import pdfplumber
            with pdfplumber.open(str(path)) as pdf:
                for i, page in enumerate(pdf.pages, 1):
                    try:
                        for t in page.extract_tables():
                            rows = [" | ".join((c or "").replace("\n", " ").strip() for c in r) for r in t if any(r)]
                            if len(rows) >= 2:
                                out.append(Document(page_content="Table:\n" + "\n".join(rows),
                                                    metadata={"source": path.name, "page": i}))
                    except Exception:
                        pass
        except Exception as e:
            print("table extraction skipped:", e)
        return out

    def _rebuild_bm25(self):
        self.bm25 = BM25Retriever.from_documents(self.chunks, preprocess_func=tokenize)
        self.bm25.k = self.cfg["k_bm25"]

    def add_pdfs(self, paths):
        new = []
        for p in map(Path, paths):
            if p.name in self.sources:
                print("already indexed:", p.name); continue
            pages = self._load(p)
            if not pages:
                print("no text found (scanned pdf?):", p.name); continue
            ch = [c for c in self.splitter.split_documents(pages) if not is_reference_chunk(c.page_content)]
            title = p.stem.replace("_", " ")
            for c in ch:
                c.page_content = f"[Document: {title}]\n{c.page_content}"
                c.metadata["chunk_id"] = len(self.chunks) + len(new)
                new.append(c)
            self.sources.append(p.name)
            print(f"{p.name}: {len(pages)} pages -> {len(ch)} chunks")
        if not new:
            return
        if self.vs is None:
            self.vs = FAISS.from_documents(new, self.emb)
        else:
            self.vs.add_documents(new)
        self.chunks += new
        self._rebuild_bm25()

    # retrieval
    def named_sources(self, query):
        q = query.lower()
        return [s for s in self.sources if any(a in q for a in self.aliases.get(s, []))]

    def _hybrid(self, query):
        dense = self.vs.similarity_search(query, k=self.cfg["k_dense"])
        sparse = self.bm25.invoke(query)
        scores, docs = {}, {}
        for lst in (dense, sparse):
            for rank, d in enumerate(lst):
                cid = d.metadata["chunk_id"]
                docs[cid] = d
                scores[cid] = scores.get(cid, 0.0) + 1.0 / (60 + rank + 1)   # RRF
        top = sorted(scores, key=scores.get, reverse=True)[:self.cfg["pool"]]
        return [docs[c] for c in top]

    def _in_source(self, query, source, k):
        return self.vs.similarity_search(query, k=k, filter={"source": source}, fetch_k=len(self.chunks))

    def _per_source(self, query, named, per=2):
        # best chunks from each document, so one paper cannot crowd out the others
        best = {}
        for s in self.sources:
            cands = self._in_source(query, s, per * 4)
            if not cands:
                continue
            ce = self.reranker.predict([(query, d.page_content) for d in cands])
            ranked = sorted(zip(cands, ce), key=lambda x: x[1], reverse=True)[:per]
            best[s] = [(d, float(x)) for d, x in ranked]
        others = sorted([s for s in best if s not in named], key=lambda s: best[s][0][1], reverse=True)
        chosen = named if len(named) >= 2 else named + others[:3 - len(named)]
        out = [h for s in chosen if s in best for h in best[s]]
        return sorted(out, key=lambda x: x[1], reverse=True)[:max(self.cfg["k_final"] + 3, per * len(chosen))]

    def retrieve(self, query):
        # returns [(chunk, rerank_score)], best first
        if self.vs is None:
            return []
        named = self.named_sources(query)
        if MULTI_RE.search(query) or len(named) > 1:
            return self._per_source(query, named)
        if len(named) == 1:                # question is about one specific document
            cands = self._in_source(query, named[0], self.cfg["pool"])
        else:
            cands = self._hybrid(query)
        ce = self.reranker.predict([(query, d.page_content) for d in cands])
        ranked = sorted(zip(cands, ce), key=lambda x: x[1], reverse=True)[:self.cfg["k_final"]]
        return [(d, float(s)) for d, s in ranked]

    # save / load for the vs code version
    def save(self, folder):
        folder.mkdir(parents=True, exist_ok=True)
        self.vs.save_local(str(folder / "faiss_index"))
        (folder / "chunks.json").write_text(json.dumps(
            [dict(text=c.page_content, **c.metadata) for c in self.chunks]))
        (folder / "config.json").write_text(json.dumps(self.cfg, indent=2))

    @classmethod
    def load(cls, folder, emb, reranker, cfg):
        idx = cls(emb, reranker, cfg)
        idx.vs = FAISS.load_local(str(folder / "faiss_index"), emb, allow_dangerous_deserialization=True)
        for r in json.loads((folder / "chunks.json").read_text()):
            text = r.pop("text")
            idx.chunks.append(Document(page_content=text, metadata=r))
        idx.sources = sorted({c.metadata["source"] for c in idx.chunks})
        idx._rebuild_bm25()
        return idx


index = DocumentIndex(emb, reranker, CFG)
index.add_pdfs(pdfs)
print(f"\n{len(index.sources)} documents, {len(index.chunks)} chunks\n")

for d, s in index.retrieve("How many attention heads does the Transformer use?")[:3]:
    print(f"{s:6.2f} | {d.metadata['source']} p.{d.metadata['page']} | {d.page_content[:90]!r}")

acme_annual_report.pdf: 3 pages -> 3 chunks
attention_is_all_you_need.pdf: 22 pages -> 47 chunks


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


llama2_technical_report.pdf: 90 pages -> 323 chunks
lora_hu_2021.pdf: 27 pages -> 97 chunks
rag_lewis_2020.pdf: 25 pages -> 77 chunks
rag_survey_gao_2023.pdf: 21 pages -> 94 chunks

6 documents, 641 chunks

  4.49 | attention_is_all_you_need.pdf p.5 | '[Document: attention is all you need]\n3.2.3 Applications of Attention in our Model\nThe Tra'
  2.70 | attention_is_all_you_need.pdf p.5 | '[Document: attention is all you need]\noutput values. These are concatenated and once again'
  1.32 | attention_is_all_you_need.pdf p.3 | '[Document: attention is all you need]\nFigure 1: The Transformer - model architecture.\nThe '


## 5. LLM (Qwen2.5-3B, fp16)

In [65]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CFG["llm_model"])
llm = AutoModelForCausalLM.from_pretrained(
    CFG["llm_model"],
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map={"": 0} if device == "cuda" else None)
llm.eval()

def chat(messages, max_new_tokens=None):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens or CFG["max_new_tokens"],
                           do_sample=False, temperature=None, top_p=None, top_k=None,
                           repetition_penalty=1.05)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(chat([{"role": "user", "content": "Say hello in one short sentence."}], 30))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Hello!


## 6. Conversational RAG

In [66]:
from langchain_core.prompts import PromptTemplate

SYSTEM_PROMPT = f"""You are a precise document assistant. Answer ONLY using the numbered context passages.
Rules:
1. Cite the passages you used inline like [1] or [2][3].
2. If the context does not contain the answer, reply exactly: "{NOT_FOUND_MSG}"
3. Never use outside knowledge. Do not guess numbers.
4. Answer in 1-4 sentences. Do NOT quote long passages and do NOT mention page numbers.
5. For comparison questions, say what each document says. Name each document (see its source) and cite it.
6. Conversation history is only for understanding the question. Facts must come from the passages."""

ANSWER_TMPL = PromptTemplate.from_template(
    "{memory}Context passages:\n{context}\n\nQuestion: {question}\n\nAnswer (with [n] citations):")

CONDENSE_TMPL = PromptTemplate.from_template(
    "Rewrite the follow-up question as a standalone question. Only replace words like 'it', 'they', "
    "'that' using the conversation. Do NOT add topics or details that are not in the follow-up. "
    "If it is already standalone, return it unchanged. Output only the question.\n\n"
    "Example:\nUser: What is LoRA?\nAssistant: A low-rank adaptation method.\n"
    "Follow-up: How many parameters does it train?\nStandalone: How many parameters does LoRA train?\n\n"
    "{memory}Follow-up: {question}\nStandalone:")

VERIFY_TMPL = PromptTemplate.from_template(
    "Passages:\n{context}\n\nQuestion: {question}\n\n"
    "Reply YES if the passages contain the answer to the question, otherwise reply NO.")

_ids = {}
def yes_prob(context, question):
    """P(YES) from the first-token logits, a score we can tune instead of a hard verdict."""
    if not _ids:
        first = lambda w: tok.encode(w, add_special_tokens=False)[0]
        _ids["yes"] = sorted({first(w) for w in ("YES", "Yes", "yes")})
        _ids["no"] = sorted({first(w) for w in ("NO", "No", "no")})
    msg = [{"role": "user", "content": VERIFY_TMPL.format(context=context, question=question)}]
    prompt = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        logits = llm(**inputs).logits[0, -1].float()
    return torch.sigmoid(torch.logsumexp(logits[_ids["yes"]], 0) - torch.logsumexp(logits[_ids["no"]], 0)).item()

CITE_RE = re.compile(r"\[(\d+(?:\s*,\s*\d+)*)\]")
REF_RE = re.compile(r"\b(it|its|they|them|their|this|that|these|those|former|latter|same|above|earlier|previous)\b", re.I)

def strip_cites(s):
    # old [n] point to old passages, so drop them before saving to memory
    s = CITE_RE.sub("", s)
    s = re.sub(r"\s+([.,;:])", r"\1", s)
    return re.sub(r"\s{2,}", " ", s).strip()


class ConversationalRAG:
    def __init__(self, index, chat_fn, cfg, yes_fn=None, not_found=NOT_FOUND_MSG):
        self.index, self.chat, self.cfg, self.yes_fn, self.not_found = index, chat_fn, cfg, yes_fn, not_found
        self.history, self.summary = [], ""

    def reset(self):
        self.history, self.summary = [], ""

    def _memory_text(self):
        parts = []
        if self.summary:
            parts.append(f"Summary of earlier conversation: {self.summary}\n")
        if self.history:
            turns = "\n".join(f"User: {q}\nAssistant: {a}" for q, a in self.history)
            parts.append(f"Recent conversation:\n{turns}\n")
        return "\n".join(parts) + ("\n" if parts else "")

    def _condense(self, question):
        if not self.history and not self.summary:
            return question
        if not REF_RE.search(question) and len(question.split()) > 5:
            return question                # nothing to resolve, skip the llm call
        p = CONDENSE_TMPL.format(memory=self._memory_text(), question=question)
        out = self.chat([{"role": "user", "content": p}], 80).split("\n")[0].strip()
        return out if 5 < len(out) < 400 else question

    def _remember(self, q, a):
        self.history.append((q, strip_cites(a)))
        while len(self.history) > self.cfg["keep_turns"]:
            oq, oa = self.history.pop(0)
            p = (f"Current summary: {self.summary or '(none)'}\n\nNew exchange:\nUser: {oq}\n"
                 f"Assistant: {oa}\n\nWrite an updated summary in at most 3 sentences, "
                 "keeping names, numbers and topics.")
            self.summary = self.chat([{"role": "user", "content": p}], 120)

    def ask(self, question, verbose=True):
        t0 = time.time()
        standalone = self._condense(question)
        hits = [(d, s) for d, s in self.index.retrieve(standalone) if s >= self.cfg["rerank_min_score"]]
        answer, sources, top, p_yes = self.not_found, [], None, None

        if hits:                           # no hits = nothing relevant, skip the llm
            top = hits[0][1]
            context = "\n\n".join(
                f"[{i}] (source: {d.metadata['source']}, page {d.metadata['page']})\n{d.page_content}"
                for i, (d, _) in enumerate(hits, 1))
            named = self.index.named_sources(standalone)
            multi = bool(MULTI_RE.search(standalone)) or len(named) > 1   # comparisons span documents
            if self.yes_fn:
                p_yes = self.yes_fn(context, standalone)
            blocked = self.cfg.get("verify") and not multi and p_yes is not None and p_yes < self.cfg["verify_min"]
            if not blocked:                # (the confidence check is skipped for comparison questions)
                q_llm = standalone
                if len(named) > 1:         # comparison: make the model cover every document
                    q_llm += " Answer with one separate sentence for each of these documents: " + ", ".join(named) + "."
                user_msg = ANSWER_TMPL.format(memory=self._memory_text(), context=context, question=q_llm)
                answer = self.chat([{"role": "system", "content": SYSTEM_PROMPT},
                                    {"role": "user", "content": user_msg}])
                cited = sorted({int(n) for m in CITE_RE.findall(answer) for n in re.split(r"\s*,\s*", m)
                                if 1 <= int(n) <= len(hits)})
                used = [hits[i - 1][0] for i in cited] if cited else [h[0] for h in hits[:3]]
                sources = sorted({(d.metadata["source"], d.metadata["page"]) for d in used})
                if self.not_found.lower().rstrip(".") in answer.lower():
                    answer, sources = self.not_found, []

        self._remember(question, answer)
        res = dict(question=question, standalone=standalone, answer=answer,
                   sources=sources, top_score=top, p_yes=p_yes, seconds=round(time.time() - t0, 1))
        if verbose:
            print(f"Q: {question}")
            if standalone != question: print(f"   (rewritten: {standalone})")
            print(f"A: {answer}")
            if sources: print("   Sources:", "; ".join(f"{s} p.{p}" for s, p in sources))
            print(f"   [{res['seconds']}s | top rerank score: {top}]\n")
        return res


rag = ConversationalRAG(index, chat, CFG, yes_fn=yes_prob)

## 7. Demo: follow-up questions

In [67]:
rag.reset()
rag.ask("What is the Transformer architecture and what does it use instead of recurrence?")
rag.ask("How many attention heads does the base model use?")
rag.ask("What are the two RAG formulations in the original RAG paper?")
rag.ask("Which of them generates each token from a different document?")
_ = rag.ask("And how does LoRA reduce the number of trainable parameters?")
print("summary of older turns:", rag.summary)

Q: What is the Transformer architecture and what does it use instead of recurrence?
A: The Transformer architecture uses stacked self-attention mechanisms instead of recurrence. It relies entirely on these mechanisms to compute representations of its input and output, avoiding sequence-aligned RNNs or convolution. This was proposed in [4] and described in detail in [1].
   Sources: attention_is_all_you_need.pdf p.2
   [3.9s | top rerank score: 2.1039633750915527]

Q: How many attention heads does the base model use?
A: According to [1], the Transformer uses 8 parallel attention layers, or heads. Therefore, the base model uses 8 attention heads.
   Sources: attention_is_all_you_need.pdf p.5
   [3.1s | top rerank score: 3.4033448696136475]

Q: What are the two RAG formulations in the original RAG paper?
A: The two RAG formulations mentioned in the original RAG paper are RAG-Token and RAG-Sequence. According to [1], the RAG-Sequence model uses the same retrieved document to generate the c

## 8. Demo: comparison and unanswerable questions

In [68]:
rag.reset()
rag.ask("Compare how the RAG survey and the original RAG paper describe the retrieval step.")
rag.reset()
rag.ask("What was Tesla's total revenue in 2023?")
_ = rag.ask("Who won the 2022 FIFA World Cup?")

Q: Compare how the RAG survey and the original RAG paper describe the retrieval step.
A: [rag_lewis_2020.pdf] RAG demonstrates strong results without requiring expensive pre-training and utilizes a retriever initialized with DPR's, showing improved performance compared to systems using BERT-based re-ranking and extractive reading.
[rag_survey_gao_2023.pdf] The retrieval step involves transforming user queries into vector representations and computing similarity scores with the indexed corpus to select the most relevant chunks for generating a coherent prompt.
   Sources: rag_lewis_2020.pdf p.5; rag_survey_gao_2023.pdf p.3; rag_survey_gao_2023.pdf p.9
   [5.9s | top rerank score: 4.745622158050537]

Q: What was Tesla's total revenue in 2023?
A: I could not find this information in the uploaded documents.
   [0.2s | top rerank score: 1.226455807685852]

Q: Who won the 2022 FIFA World Cup?
A: I could not find this information in the uploaded documents.
   [0.1s | top rerank score: None]



## 9. Evaluation

In [69]:
import pandas as pd

# kind: answerable / near (topic is in the documents but the answer is not) / off (unrelated)
EVAL = [
    dict(kind="answerable", q="How many attention heads does the base Transformer use?", src=["attention"], kw=("8", "eight")),
    dict(kind="answerable", q="How many identical layers does the Transformer encoder have?", src=["attention"], kw=("6", "six")),
    dict(kind="answerable", q="What retriever is used in the original RAG paper?", src=["rag_lewis"], kw=("dpr", "dense passage")),
    dict(kind="answerable", q="What are the two RAG formulations in the original RAG paper?", src=["rag_lewis"], kw=("token",)),
    dict(kind="answerable", q="What does LoRA freeze and what does it train?", src=["lora"], kw=("rank",)),
    dict(kind="answerable", q="By how much does LoRA reduce trainable parameters compared to full fine-tuning of GPT-3 175B?", src=["lora"], kw=("10,000", "10000")),
    dict(kind="answerable", q="What range of parameter sizes does Llama 2 cover?", src=["llama2"], kw=("70",)),
    dict(kind="answerable", q="What is Llama 2-Chat optimized for?", src=["llama2"], kw=("dialogue",)),
    dict(kind="answerable", q="What are Naive RAG, Advanced RAG and Modular RAG?", src=["rag_survey"], kw=("modular",)),
    dict(kind="answerable", q="According to the RAG survey, what are the main components of the RAG framework?", src=["rag_survey"], kw=("augment",)),
    dict(kind="answerable", q="Compare how the original RAG paper and the RAG survey describe retrieval.", src=["rag_lewis", "rag_survey"], kw=("retriev",)),
    dict(kind="answerable", q="What was Acme's net profit in 2023?", src=["acme"], kw=("331",)),
    dict(kind="answerable", q="What was Acme's EBITDA margin in 2022?", src=["acme"], kw=("15.1",)),
    dict(kind="answerable", q="Who is the CEO of Acme?", src=["acme"], kw=("meera",)),
    dict(kind="near", q="How many parameters does GPT-4 have?"),
    dict(kind="near", q="What BLEU score did the RAG model get on German to English translation?"),
    dict(kind="near", q="Who is the CEO of Meta?"),
    dict(kind="near", q="What was Acme's net profit in 2019?"),
    dict(kind="off", q="What was Tesla's total revenue in 2023?"),
    dict(kind="off", q="What is the capital of Australia?"),
]
rows = []
for e in EVAL:
    rag.reset()
    r = rag.ask(e["q"], verbose=False)
    row = dict(kind=e["kind"], q=e["q"][:60], refused=NOT_FOUND_MSG in r["answer"], top_score=r["top_score"], p_yes=r["p_yes"], answer=r["answer"][:80])
    if e["kind"] == "answerable":
        row["src_hit"] = all(any(x in s for s, _ in r["sources"]) for x in e["src"])   # every expected doc cited
        row["kw_hit"] = any(k in r["answer"].lower() for k in e["kw"])
    rows.append(row)

df = pd.DataFrame(rows)
display(df)
ans, no = df[df.kind == "answerable"], df[df.kind != "answerable"]
print(f"Answerable ({len(ans)}): source hit {ans.src_hit.astype(float).mean():.0%} | keyword hit {ans.kw_hit.astype(float).mean():.0%} | wrongly refused {ans.refused.mean():.0%}")
print(f"Unanswerable ({len(no)}): correct refusals {no.refused.mean():.0%} (near-miss {df[df.kind == 'near'].refused.mean():.0%})")

# can p_yes separate answerable from unanswerable? (needs verify off to see every answer)
lo, hi = ans.p_yes.min(), no.p_yes.max()
if pd.notna(lo) and pd.notna(hi):
    print(f"p_yes: answerable min {lo:.2f} | unanswerable max {hi:.2f} ->",
          f"separable, verify_min between {max(hi, 1e-12):.0e} and {lo:.2f} is safe" if lo > hi else "overlaps, keep verify off")

# memory check: does the follow-up get rewritten with the right topic?
rag.reset()
rag.ask("What is LoRA?", verbose=False)
r = rag.ask("How does it reduce trainable parameters?", verbose=False)
print("memory ok:", "lora" in r["standalone"].lower(), "|", r["standalone"])

,kind,q,refused,top_score,p_yes,answer,src_hit,kw_hit
0,answerable,How many attention heads does the base Transfo...,False,2.703742,9.697856e-01,The base Transformer uses 8 parallel attention...,True,True
1,answerable,How many identical layers does the Transformer...,False,7.496346,9.999998e-01,The Transformer encoder has 6 identical layers...,True,True
2,answerable,What retriever is used in the original RAG paper?,False,3.793722,9.992679e-01,The original RAG paper uses the Dense Passage ...,True,True
3,answerable,What are the two RAG formulations in the origi...,False,-0.861886,9.999968e-01,The two RAG formulations in the original RAG p...,True,True
4,answerable,What does LoRA freeze and what does it train?,False,3.240872,9.999697e-01,LoRA freezes the pre-trained model weights [1]...,True,True
5,answerable,By how much does LoRA reduce trainable paramet...,False,7.175242,9.999877e-01,LoRA reduces the number of trainable parameter...,True,True
6,answerable,What range of parameter sizes does Llama 2 cover?,False,6.869830,9.999985e-01,Llama 2 covers parameter sizes ranging from 7B...,True,True
7,answerable,What is Llama 2-Chat optimized for?,False,7.348386,9.999982e-01,Llama 2-Chat is optimized for dialogue use cas...,True,True
8,answerable,"What are Naive RAG, Advanced RAG and Modular RAG?",False,8.946070,9.999948e-01,"Naive RAG represents the earliest methodology,...",True,True
9,answerable,"According to the RAG survey, what are the main...",False,6.931486,9.999938e-01,"According to the RAG survey, the main componen...",True,True


Answerable (14): source hit 100% | keyword hit 100% | wrongly refused 0%
Unanswerable (6): correct refusals 100% (near-miss 100%)
p_yes: answerable min 0.04 | unanswerable max 0.00 -> separable, verify_min between 2e-09 and 0.04 is safe
memory ok: True | How does LoRA reduce trainable parameters?


In [70]:
index.save(STORE_DIR)
shutil.make_archive(str(WORK / "rag_store"), "zip", STORE_DIR)
print("Saved:", WORK / "rag_store.zip")

# check it reloads
index2 = DocumentIndex.load(STORE_DIR, emb, reranker, CFG)
print(len(index2.chunks), "chunks reloaded |", index2.retrieve("What is LoRA?")[0][0].metadata)

Saved: /kaggle/working/rag_store.zip
641 chunks reloaded | {'source': 'lora_hu_2021.pdf', 'page': 1, 'chunk_id': 373}
